# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and records
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s.

In [ ]:
# List available record sets
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print("No record sets found in the metadata. The dataset structure may be nested inside distributions or provided via linked files.")
    # Try to get record sets from distributions
    distributions = getattr(metadata, 'distribution', [])
    all_record_sets = []
    for dist in distributions:
        dist_obj = dist
        # mlcroissant handles linked resources
        record_sets_inner = getattr(dist_obj, 'recordSet', [])
        if record_sets_inner:
            all_record_sets.extend(record_sets_inner)
    record_sets = all_record_sets

if not record_sets:
    print("No record sets found in either top-level or distribution. Please check the Croissant schema.")
else:
    for i, rs in enumerate(record_sets):
        print(f"Record Set {i+1} @id: {getattr(rs, '@id', str(rs))}")
        print(f"  Name: {getattr(rs, 'name', 'N/A')}")
        print(f"  Description: {getattr(rs, 'description', 'N/A')}")
        # Fields
        fields = getattr(rs, 'field', [])
        if fields:
            for f in fields:
                print(f"    Field: {getattr(f, 'name', 'N/A')} (@id={getattr(f, '@id', str(f))})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For the FAIR^2 dataset, the main record set contains protein data. Let's load all available record sets by their `@id`s.
# To dynamically discover all record set @ids even if the hierarchy is non-flat, recreate the step from above:

def collect_recordset_ids(metadata):
    ids = []
    record_sets = getattr(metadata, 'recordSet', [])
    if record_sets:
        for rs in record_sets:
            ids.append(getattr(rs, '@id', str(rs)))
    else:
        # Try distribution
        distributions = getattr(metadata, 'distribution', [])
        for dist in distributions:
            for rs in getattr(dist, 'recordSet', []):
                ids.append(getattr(rs, '@id', str(rs)))
    return ids

# Collect @id for all record sets
record_set_ids = collect_recordset_ids(metadata)

if not record_set_ids:
    raise Exception("No record sets (@id) found in dataset metadata.")

# Load all record sets into a dictionary of DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# Choose one of the discovered record sets as the main table for analysis
main_record_set_id = record_set_ids[0]  # or choose from list if >1
print(f"\nMain analysis record set: {main_record_set_id}")
print("Columns:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes in preparation for further analysis.

In [ ]:
# List numeric columns
df = dataframes[main_record_set_id]
print("Numeric fields detected:", df.select_dtypes(include=['number']).columns.tolist())

# For example, let's pick a column named 'Coverage [%]' or 'MW [kDa]' for filtering if present
possible_numeric_fields = ['Coverage [%]', 'MW [kDa]', 'pI', 'Sum PEP Score', 'Unique Peptide Count', 'Protein Score']
numeric_field = None
for f in possible_numeric_fields:
    if f in df.columns:
        numeric_field = f
        break

if numeric_field is None:
    print("No expected numeric fields found. Please check available columns and choose a numeric column for analysis.")
    numeric_field = df.select_dtypes(include=['number']).columns[0] if not df.select_dtypes(include=['number']).empty else None

if numeric_field:
    threshold = df[numeric_field].mean()
    print(f"Filtering with {numeric_field} > {threshold:.2f}")
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Grouping by, e.g., 'Sample' column if exists
    possible_group_fields = ['Sample', 'Condition', 'Protein Class', 'Experiment']
    group_field = None
    for g in possible_group_fields:
        if g in filtered_df.columns:
            group_field = g
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we have loaded and explored the FAIR^2 mass spectrometry dataset using its Croissant schema and the `mlcroissant` library. We identified available record sets by their `@id`, loaded the main record set into a DataFrame, previewed numeric data columns, and performed basic EDA and visualization. This approach provides a reproducible way to interact with FAIR datasets defined by Croissant.

Further analysis might include advanced statistics, protein function analysis, or modeling, depending on your research question.